In [ ]:
import xml.etree.ElementTree as ET
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.lines import Line2D

# ==============================================================================
# 1. PARSE XML
# ==============================================================================
def parse_electrode_xml(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    electrodes = {}
    for elec in root.iter("Electrode"):
        label = elec.find("Label").text
        x = float(elec.find("XCoordinate").text)
        y = float(elec.find("YCoordinate").text)
        group_id = elec.find("GroupId").text
        electrodes[label] = {"x": x, "y": y, "group": group_id}
    return electrodes


xml_path = "SEEG Hanna.xml"
electrodes = parse_electrode_xml(xml_path)
print(f"Parsed {len(electrodes)} electrode contacts")

# ==============================================================================
# 2. AUTO-FIT HEAD CIRCLE TO ACTUAL DATA BOUNDS
# ==============================================================================
xs = [v["x"] for v in electrodes.values()]
ys = [v["y"] for v in electrodes.values()]

data_cx = (min(xs) + max(xs)) / 2
data_cy = (min(ys) + max(ys)) / 2
half_width = (max(xs) - min(xs)) / 2
half_height = (max(ys) - min(ys)) / 2

# Head radius = distance to the farthest point, plus 20% padding
radius = max(half_width, half_height) * 1.35

# ==============================================================================
# 3. COLOR PER SHAFT (auto-pick colormap based on number of groups)
# ==============================================================================
groups = sorted(set(v["group"] for v in electrodes.values()), key=int)
n_groups = len(groups)
cmap = plt.colormaps["tab20"].resampled(n_groups) if n_groups <= 20 else plt.colormaps["hsv"].resampled(n_groups)
group_color = {g: cmap(i) for i, g in enumerate(groups)}

# ==============================================================================
# 4. DRAW HEAD OUTLINE
# ==============================================================================
def draw_head_outline(ax, center, radius):
    cx, cy = center
    head = Circle((cx, cy), radius, fill=False, linewidth=2, zorder=1)
    ax.add_patch(head)

    nose_width = radius * 0.15
    nose_height = radius * 0.10
    nose = Line2D(
        [cx - nose_width, cx, cx + nose_width],
        [cy - radius, cy - radius - nose_height, cy - radius],
        color="black", linewidth=2, zorder=1
    )
    ax.add_line(nose)

    ear_width = radius * 0.06
    ear_height = radius * 0.20
    for side in [-1, 1]:
        ear_x = cx + side * radius
        ear = Line2D(
            [ear_x, ear_x + side * ear_width, ear_x + side * ear_width, ear_x],
            [cy - ear_height / 2, cy - ear_height / 2, cy + ear_height / 2, cy + ear_height / 2],
            color="black", linewidth=2, zorder=1
        )
        ax.add_line(ear)


# ==============================================================================
# 5. PLOT
# ==============================================================================
fig, ax = plt.subplots(figsize=(16, 16))

draw_head_outline(ax, center=(data_cx, data_cy), radius=radius)

for label, v in electrodes.items():
    x, y = v["x"], v["y"]
    color = group_color[v["group"]]
    ax.scatter(x, y, s=45, color=color, zorder=3, edgecolors="black", linewidths=0.5)
    ax.annotate(
        label, (x, y),
        textcoords="offset points", xytext=(0, 6),
        fontsize=6, rotation=45, ha="left", va="bottom", zorder=4
    )

ax.set_title(f"SEEG Electrode Placement ({len(electrodes)} contacts)", fontsize=14)
ax.invert_yaxis()
ax.set_aspect("equal")

margin = radius * 0.25
ax.set_xlim(data_cx - radius - margin, data_cx + radius + margin)
ax.set_ylim(data_cy + radius + margin, data_cy - radius - margin - radius * 0.15)  # extra room for nose
ax.axis("off")

plt.tight_layout()
plt.savefig("electrode_placement_head.png", dpi=300, bbox_inches="tight")
plt.show()